# Round 2 Strategy: ATR Cross-Sectional Alpha

**Pipeline:**
1. Load data (features, universe, returns)
2. Cross-sectional z-score of ATR + clip outliers
3. Construct targets: forward log returns → strip beta → z-score
4. Train PyGAM with monotonic constraint
5. Generate portfolio weights (vol-scaled, dollar-neutral)
6. Backtest using QRT `utils.py`

In [ ]:
import pandas as pd
import numpy as np
import os, sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
from utils import backtest_portfolio, scale_to_book_long_short

DATA_DIR = 'stores'

# Prediction horizon (trading days)
H = 5

# Rolling beta window
BETA_WINDOW = 60

# Train/val split ratio
TRAIN_RATIO = 0.70

# Volatility lookback for position sizing
VOL_WINDOW = 20

# Z-score clip threshold
ZSCORE_CLIP = 3.0

## 1. Load Data

In [ ]:
# Load features (MultiIndex: level 0 = indicator name, level 1 = ticker)
features_df = pd.read_parquet(os.path.join(DATA_DIR, 'features.parquet'))
atr = features_df['average_true_range']  # shape: (dates, tickers)

# Load universe mask (1 = in universe, 0 = out)
universe = pd.read_parquet(os.path.join(DATA_DIR, 'universe_5m.parquet'))

# Load returns
returns = pd.read_parquet(os.path.join(DATA_DIR, 'returns.parquet'))

# Align all three on common dates and tickers
common_dates = atr.index.intersection(universe.index).intersection(returns.index)
common_tickers = atr.columns.intersection(universe.columns).intersection(returns.columns)

atr = atr.loc[common_dates, common_tickers].astype('float64')
universe = universe.loc[common_dates, common_tickers]
returns = returns.loc[common_dates, common_tickers].astype('float64')

print(f'Dates: {len(common_dates)}, Tickers: {len(common_tickers)}')
print(f'Date range: {common_dates[0].date()} → {common_dates[-1].date()}')

## 2. Feature Engineering: Cross-Sectional Z-Score of ATR

In [ ]:
# Mask ATR by universe (out-of-universe → NaN)
atr_masked = atr.where(universe == 1)

# Cross-sectional z-score: at each date t, standardize across all stocks
cs_mean = atr_masked.mean(axis=1)  # mean across stocks for each date
cs_std = atr_masked.std(axis=1)    # std across stocks for each date

# Z-score: (ATR_i,t - mu_t) / sigma_t
X = atr_masked.sub(cs_mean, axis=0).div(cs_std, axis=0)

# Clip outliers at ±3 sigma
X = X.clip(-ZSCORE_CLIP, ZSCORE_CLIP)

print(f'Feature X shape: {X.shape}')
print(f'X range: [{X.min().min():.2f}, {X.max().max():.2f}]')
print(f'NaN fraction: {X.isna().sum().sum() / X.size:.2%}')

## 3. Target Construction

### 3a. Forward Log Returns

In [ ]:
# Use Adj Close prices to compute forward log returns
pv = pd.read_pickle('top_5000_yf_data.pkl')
adj_close = pv['Adj Close'][common_tickers].loc[common_dates].astype('float64')

# h-day forward log return: ln(P_{t+h} / P_t)
fwd_log_ret = np.log(adj_close.shift(-H) / adj_close)

print(f'Forward log return shape: {fwd_log_ret.shape}')
print(f'Non-NaN: {fwd_log_ret.notna().sum().sum()}')

# Free memory
del pv

### 3b. Beta Stripping

Rolling OLS to estimate beta, then subtract market component from forward returns.

In [ ]:
# Market return = equal-weighted average of all in-universe stock returns
returns_masked = returns.where(universe == 1)
market_ret = returns_masked.mean(axis=1)

# Rolling beta via vectorized rolling cov / rolling var
# beta_i = cov(r_i, r_m) / var(r_m) over BETA_WINDOW days

# Expand market_ret to match returns shape for element-wise ops
rolling_cov = returns_masked.rolling(BETA_WINDOW, min_periods=30).apply(
    lambda x: np.nan, raw=True)  # placeholder - replaced below

# More efficient: compute rolling stats manually
print('Computing rolling betas...')

# For each stock, rolling covariance with market
rolling_var_m = market_ret.rolling(BETA_WINDOW, min_periods=30).var()

betas = pd.DataFrame(index=common_dates, columns=common_tickers, dtype='float64')

for ticker in common_tickers:
    stock_ret = returns_masked[ticker]
    # Rolling covariance: cov(stock, market)
    # Using the identity: cov(X,Y) = E[XY] - E[X]E[Y]
    xy = stock_ret * market_ret
    roll_mean_xy = xy.rolling(BETA_WINDOW, min_periods=30).mean()
    roll_mean_x = stock_ret.rolling(BETA_WINDOW, min_periods=30).mean()
    roll_mean_y = market_ret.rolling(BETA_WINDOW, min_periods=30).mean()
    roll_cov = roll_mean_xy - roll_mean_x * roll_mean_y
    betas[ticker] = roll_cov / rolling_var_m

print(f'Betas computed. Shape: {betas.shape}')
print(f'Median beta (last date): {betas.iloc[-1].median():.3f}')

In [ ]:
# Compute h-day forward market return for stripping
fwd_market_ret = market_ret.rolling(H).sum().shift(-H)

# Residual forward return = raw forward return - beta * forward market return
residual_fwd_ret = fwd_log_ret.sub(betas.mul(fwd_market_ret, axis=0))

print(f'Residual forward return shape: {residual_fwd_ret.shape}')

### 3c. Cross-Sectional Z-Score of Target

In [ ]:
# Mask by universe
residual_fwd_ret_masked = residual_fwd_ret.where(universe == 1)

# Cross-sectional z-score of target
target_mean = residual_fwd_ret_masked.mean(axis=1)
target_std = residual_fwd_ret_masked.std(axis=1)

Y = residual_fwd_ret_masked.sub(target_mean, axis=0).div(target_std, axis=0)

# Clip target outliers too
Y = Y.clip(-ZSCORE_CLIP, ZSCORE_CLIP)

print(f'Target Y shape: {Y.shape}')

## 4. Model Training: PyGAM with Monotonic Constraint

Train a `LinearGAM` with `s(0, constraints='monotonic_inc')` to learn the nonlinear mapping from standardized ATR to expected residual return.

In [ ]:
from pygam import LinearGAM, s

# Flatten X and Y into paired observations, dropping NaNs
# Only use dates where both X and Y are valid
n_dates = len(common_dates)
train_end_idx = int(n_dates * TRAIN_RATIO)
embargo_end_idx = train_end_idx + H  # embargo gap

train_dates = common_dates[:train_end_idx]
val_dates = common_dates[embargo_end_idx:]

print(f'Train: {train_dates[0].date()} → {train_dates[-1].date()} ({len(train_dates)} days)')
print(f'Embargo: {H} days')
print(f'Validation: {val_dates[0].date()} → {val_dates[-1].date()} ({len(val_dates)} days)')

In [ ]:
# Build training data
X_train = X.loc[train_dates].values.flatten()
Y_train = Y.loc[train_dates].values.flatten()

# Remove NaNs
mask = ~(np.isnan(X_train) | np.isnan(Y_train))
X_train_clean = X_train[mask].reshape(-1, 1)
Y_train_clean = Y_train[mask]

print(f'Training samples: {len(X_train_clean)}')

# Subsample if too large (PyGAM can be slow on millions of points)
MAX_SAMPLES = 500_000
if len(X_train_clean) > MAX_SAMPLES:
    idx = np.random.choice(len(X_train_clean), MAX_SAMPLES, replace=False)
    X_train_sub = X_train_clean[idx]
    Y_train_sub = Y_train_clean[idx]
    print(f'Subsampled to {MAX_SAMPLES} for training')
else:
    X_train_sub = X_train_clean
    Y_train_sub = Y_train_clean

In [ ]:
# Fit GAM with monotonically increasing spline
print('Fitting LinearGAM with monotonic_inc constraint...')
gam = LinearGAM(s(0, constraints='monotonic_inc'))
gam.gridsearch(X_train_sub, Y_train_sub, progress=True)

print(f'GAM fitted. R-squared: {gam.statistics_["pseudo_r2"]["explained_deviance"]:.4f}')
print(f'Lambda: {gam.lam}')

In [ ]:
# Evaluate on validation set: Information Coefficient (rank correlation)
from scipy.stats import spearmanr

ic_list = []
for date in val_dates:
    x_row = X.loc[date].dropna()
    y_row = Y.loc[date].dropna()
    common = x_row.index.intersection(y_row.index)
    if len(common) < 20:
        continue
    x_vals = x_row[common].values.reshape(-1, 1)
    y_vals = y_row[common].values
    y_pred = gam.predict(x_vals)
    ic, _ = spearmanr(y_pred, y_vals)
    if not np.isnan(ic):
        ic_list.append(ic)

ic_array = np.array(ic_list)
print(f'Mean IC: {ic_array.mean():.4f}')
print(f'IC Std: {ic_array.std():.4f}')
print(f'IC IR (IC/std): {ic_array.mean()/ic_array.std():.4f}')
print(f'Hit Rate (IC > 0): {(ic_array > 0).mean():.2%}')

## 5. Portfolio Construction

Generate daily portfolio weights:
1. Predict expected residual return from ATR z-score
2. Volatility-scale the alpha scores
3. Apply dollar-neutral long-short weighting

In [ ]:
# Compute trailing 20-day realized volatility for each stock
trailing_vol = returns_masked.rolling(VOL_WINDOW, min_periods=10).std()

# We build the portfolio on the validation period
portfolio = pd.DataFrame(0.0, index=val_dates, columns=common_tickers)

print('Building portfolio...')
for date in val_dates:
    # Get today's features
    x_today = X.loc[date]
    vol_today = trailing_vol.loc[date]
    univ_today = universe.loc[date]
    
    # Only in-universe stocks with valid feature and vol
    valid = (univ_today == 1) & x_today.notna() & vol_today.notna() & (vol_today > 0)
    valid_tickers = valid[valid].index
    
    if len(valid_tickers) < 20:
        continue
    
    # Predict alpha scores
    x_vals = x_today[valid_tickers].values.reshape(-1, 1)
    alpha = pd.Series(gam.predict(x_vals), index=valid_tickers)
    
    # Cross-sectional z-score the predictions
    alpha = (alpha - alpha.mean()) / alpha.std()
    
    # Volatility scaling: divide by trailing vol
    alpha = alpha / vol_today[valid_tickers]
    
    # Handle any inf/nan from division
    alpha = alpha.replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(alpha) < 20:
        continue
    
    # Dollar-neutral weighting: long book = +0.5, short book = -0.5
    weights = scale_to_book_long_short(alpha)
    
    # Cap max weight at 10% (QRT constraint)
    if weights.abs().max() > 0.1:
        weights = weights.clip(-0.1, 0.1)
        # Rescale to maintain unit capital
        weights = scale_to_book_long_short(weights)
    
    portfolio.loc[date, weights.index] = weights.values

print(f'Portfolio built. Non-zero days: {(portfolio.abs().sum(axis=1) > 0).sum()}')

## 6. Backtest

In [ ]:
# Align portfolio, returns, and universe for backtest
bt_dates = val_dates
bt_portfolio = portfolio.loc[bt_dates, common_tickers].fillna(0).astype('float64')
bt_returns = returns.loc[bt_dates, common_tickers].fillna(0).astype('float64')
bt_universe = universe.loc[bt_dates, common_tickers]

# Remove any dates where we have zero weights (no portfolio generated)
active_mask = bt_portfolio.abs().sum(axis=1) > 1e-10
bt_portfolio = bt_portfolio.loc[active_mask]
bt_returns = bt_returns.loc[active_mask]
bt_universe = bt_universe.loc[active_mask]

print(f'Backtest period: {bt_portfolio.index[0].date()} → {bt_portfolio.index[-1].date()}')
print(f'Active trading days: {len(bt_portfolio)}')
print()

# Run QRT backtester
net_sharpe, gross_pnl = backtest_portfolio(
    bt_portfolio, bt_returns, bt_universe,
    plot_=True, print_=True
)

In [ ]:
# Additional metrics
cumulative_pnl = gross_pnl.cumsum()
max_drawdown = (cumulative_pnl - cumulative_pnl.cummax()).min()
print(f'Max Drawdown: {max_drawdown:.4f}')
print(f'Final Cumulative PnL: {cumulative_pnl.iloc[-1]:.4f}')